# Lab 11 — Instrument a Training Loop
**Goal:** Measure loss, tokens/sec, elapsed time, and peak memory for a tiny training run.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
import torch, time
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)


## Prediction
What should a production training dashboard record besides loss?


In [ ]:
x=torch.randn(64,256,512,device=device)
w=torch.randn(512,512,device=device,requires_grad=True)
opt=torch.optim.AdamW([w],lr=1e-3)
if device=="cuda": torch.cuda.reset_peak_memory_stats()
t0=time.perf_counter()
for _ in range(10):
    y=x@w
    loss=(y*y).mean()
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
if device=="cuda": torch.cuda.synchronize()
elapsed=time.perf_counter()-t0
print("loss:",loss.item(),"elapsed:",elapsed)
if device=="cuda": print("peak GiB:",torch.cuda.max_memory_allocated()/1024**3)


## Interpretation
A useful dashboard includes throughput, memory, learning rate, gradient norms, data-loader timing, utilization/profiling data, checkpoint events, evaluation, and errors.


In [ ]:
tokens=10*64*256
print("tokens/s:",tokens/max(elapsed,1e-9))
print("This is a toy measurement: real LM throughput depends on model, precision, sequence length, kernels, and input pipeline.")


## Answers
Without instrumentation you cannot distinguish “model is learning slowly” from “input pipeline is starving the GPU” or “communication dominates.” Observability is part of training engineering.

### Primary source
https://cs336.stanford.edu/


## Deepening: instrument the training state

The goal is to distinguish model-compute time from data and systems overhead. Predict before executing.


In [ ]:
import time, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn(1024, 128, device=device)
w = torch.randn(128, 128, device=device, requires_grad=True)

for step in range(5):
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    y = x @ w
    loss = (y**2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 1e-3 * w.grad
        w.grad.zero_()
    if device == "cuda": torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()/2**30 if device == "cuda" else float("nan")
    print(step, "loss=", float(loss), "step_s=", time.perf_counter()-t0, "peak_GiB=", peak)


## Break experiment

Add a deliberate data-loading delay before the forward pass. Compare total step time with compute time.

Required report:

loss → step time → data wait → memory → throughput → bottleneck diagnosis

Complete templates/EXPERIMENT_CARD.md.
